In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005


config = Config()

In [4]:
print(utils.recall_vals_one_row([1, 2, 3], 2))
print(utils.recall_vals_one_row([1, 2, 3], 4))

True
False


In [5]:
utils.recall_several_targets(candidates=[[1, 2, 3], [1]], targets=[[1, 3, 2], [1, 2]])

0.8

In [6]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=16, progress_bar=True, use_memory_fs=False)

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [7]:
%%time

PATH_DATA = '../data/02_intermediate/'
df_train = pl.read_parquet(PATH_DATA + 'df_train_n_session.parquet')
df_test = pl.read_parquet(PATH_DATA + 'df_test_n_session.parquet')
# df_train = df_train.filter(pl.col("user") % 100 == 0)
# df_train = df_train.to_pandas()
# df_test = df_test.to_pandas()
# # df_test = catalog.load("df_test")

# df_train.name = "df_train"
# df_test.name = "df_test"

# df_train = utils.make_date_stats(df_train)
# df_test = utils.make_date_stats(df_test)

# df_train.drop(['date_day'], axis=1, inplace=True)
# df_test.drop(['date_day'], axis=1, inplace=True)

# df_train = pl.from_pandas(df_train)
# df_test = pl.from_pandas(df_test)

CPU times: user 11.9 s, sys: 3.45 s, total: 15.4 s
Wall time: 3.79 s


In [8]:
# X_val = pl.read_parquet("valid_ult.parquet")

In [9]:
# X_train = pl.read_parquet("train_ult.parquet")

In [10]:
# X_train_old = pl.read_parquet("train.parquet")

In [11]:
# X_train.filter((pl.col("pos_0") == 671720) & (pl.col("pos_9") == 34008.0))

In [12]:
# X_train_old

In [13]:
%%time

test_users = df_test[config.col_user].unique()
df = pl.concat([df_train, df_test])
df = df.sort(config.col_ts, reverse=True)

df = df.to_pandas()
df[config.col_num_action] = df.groupby(config.col_user)[config.col_aid].cumcount()
df = pl.from_pandas(df)

df = df.with_columns(
    [
        pl.col(config.col_num_action)
        .cast(pl.Int32, strict=True)
        .alias(config.col_num_action),
        pl.col(config.col_n_session)
        .cast(pl.Int32, strict=True)
        .alias(config.col_n_session),
    ]
)

CPU times: user 7min 32s, sys: 55.3 s, total: 8min 27s
Wall time: 1min 7s


In [14]:
del df_train, df_test
gc.collect()

2730

In [15]:
# ### IF DEBUG PIPELINE !!!

# df = df.filter(pl.col("user") % 100 == 0)

In [16]:
counts_aid = df.groupby([config.col_aid]).count().sort("count", reverse=True)

In [17]:
df.head()

user,aid,ts,type,n_session,num_action
i64,i64,i64,str,i32,i32
14571581,1100210,1662328791563,"""clicks""",0,0
14571515,781377,1662328789887,"""clicks""",0,0
14571170,336732,1662328788088,"""clicks""",0,0
14568399,104775,1662328781349,"""clicks""",0,0
14571580,202353,1662328781067,"""clicks""",0,0


In [18]:
counts_aid.head()

aid,count
i64,u32
1460571,137874
485256,135892
108125,124885
29735,116215
1733943,106512


In [19]:
df.shape

(223644219, 6)

In [20]:
df = df.to_pandas()

In [21]:
def split_data(df, validation_cut, test_users):
    df_val = df[
        (df[config.col_num_action] <= validation_cut)
        & (df[config.col_user].isin(test_users))
    ]

    df_val_seq = df[
        (df[config.col_num_action] > validation_cut)
        & (df[config.col_user].isin(test_users))
    ]

    df_train = df.drop(df_val.index)
    return df_train, df_val, df_val_seq

In [22]:
%%time

X_train, X_val, X_val_seq = split_data(df, validation_cut=2, test_users=test_users)

CPU times: user 25.3 s, sys: 2.63 s, total: 27.9 s
Wall time: 27.9 s


In [23]:
%%time

trs = 20
counts_aid_filtered = counts_aid.filter(pl.col("count") >= trs).to_pandas()
pad = len(counts_aid_filtered)

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=pad)
ordinal_encoder.fit(counts_aid_filtered[config.col_aid].values.reshape(-1, 1))

CPU times: user 603 ms, sys: 12.1 ms, total: 615 ms
Wall time: 611 ms


OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=954978)

In [24]:
%%time

X_train[config.col_aid] = ordinal_encoder.transform(
    X_train[config.col_aid].values.reshape(-1, 1)
).astype(int)
X_val_seq[config.col_aid] = ordinal_encoder.transform(
    X_val_seq[config.col_aid].values.reshape(-1, 1)
).astype(int)

CPU times: user 1min 46s, sys: 3.12 s, total: 1min 49s
Wall time: 1min 49s


In [25]:
%%time

X_train = pl.from_pandas(X_train)
X_train = X_train.sort(config.col_ts)

X_val = pl.from_pandas(X_val)
X_val = X_val.sort(config.col_ts)

X_val_seq = pl.from_pandas(X_val_seq)
X_val_seq = X_val_seq.sort(config.col_ts)

CPU times: user 3min 27s, sys: 1min 16s, total: 4min 43s
Wall time: 22.1 s


In [26]:
# X_train.filter(pl.col(config.col_user) == 0)

In [27]:
%%time

X_train = (
    X_train.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
)

X_val = (
    X_val.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

X_val_seq = (
    X_val_seq.groupby([config.col_user])
    .agg(
        [
            pl.col(config.col_aid).list(),
        ]
    )
)

CPU times: user 5min 2s, sys: 29.8 s, total: 5min 31s
Wall time: 57.4 s


In [28]:
# X_train.shape

In [29]:
# X_train.filter(pl.col(config.col_user) == 0)

In [30]:
counts_aid["aid"].min(), counts_aid["aid"].max()

(0, 1855602)

In [31]:
pad

954978

In [32]:
counts_aid.tail()

aid,count
i64,u32
738574,3
659614,3
537791,3
273615,3
148383,3


In [33]:
counts_aid.filter(pl.col("aid") == 1)

aid,count
i64,u32
1,34


In [34]:
# len(counts_aid), len(counts_aid_filtered)

In [35]:
# len(token_to_idx)

In [36]:
# min(list(token_to_idx.values()))

In [37]:
# token_to_idx[0]

In [38]:
l = [1, 2, 3]
print(utils.split_list_randomly(l=l, seed=1))
print(utils.split_list_randomly(l=l, seed=2))

([1, 2], [3])
([1], [2, 3])


In [39]:
# X_train.head()

In [40]:
# X_train.shape

In [41]:
X_val.head()

user,aid
i64,list[i64]
13534272,"[296142, 1043933, 1043933]"
13592192,[1343718]
13433024,[14332]
13401152,[690855]
13937600,"[1333440, 1616754, 1333440]"


In [42]:
%%time

X_train = X_train.to_pandas()
X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: x[:-1])
X_train[config.col_target] = X_train[config.col_aid].parallel_apply(lambda x: x[-1])
X_train = pl.from_pandas(X_train)

IOStream.flush timed out


CPU times: user 2min 34s, sys: 31.8 s, total: 3min 6s
Wall time: 3min 12s


In [43]:
# X_train.head()

In [44]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
139904,19
397824,4
868480,74
375872,9
469056,2
263040,4
4928,2
603264,6
125632,1


In [45]:
del df
gc.collect()

537

In [46]:
%%time

X_train = X_train.to_pandas()

max_len = config.max_len

list_train = list()
list_target = list()
list_user = list()

n_iter = 5
for i in tqdm(range(len(X_train))):
    row = X_train.iloc[i]
    row_train = list(row[config.col_train_seq])
    target_cur_main = row[config.col_target]
    len_row_train = len(row_train)
    if len_row_train == 1 and target_cur_main < pad:
        list_train.append(tuple(row_train))
        list_target.append(target_cur_main)
        list_user.append(row[config.col_user])
    else:
        n_iter_cur = min(n_iter, len_row_train - 1)
        train_cur_main = utils.truncate_list(l=row_train, max_len=max_len, from_start=False)
        if target_cur_main < pad:
            list_train.append(tuple(train_cur_main))
            list_target.append(target_cur_main)
            list_user.append(row[config.col_user])

        for seed in range(n_iter_cur):
            train_cur, target_cur = utils.split_list_randomly(l=row_train, seed=seed)
            targets = target_cur[:2]
            for target_cur in targets:
                if target_cur < pad:
                    train_cur = utils.truncate_list(l=train_cur, max_len=max_len, from_start=False)
                    list_train.append(tuple(train_cur))
                    list_target.append(target_cur)
                    list_user.append(row[config.col_user])
            
        # for k in range(1, len_row_train - 1):
        #     train_cur, target_cur = row_train[:k], row_train[k]
        #     if target_cur < pad:
        #         train_cur = utils.truncate_list(l=train_cur, max_len=max_len, from_start=False)
        #         list_train.append(tuple(train_cur))
        #         list_target.append(target_cur)
        #         list_user.append(row[config.col_user])

del X_train
gc.collect()

X_train = pd.DataFrame()

X_train[config.col_user] = list_user
del list_user
gc.collect()

X_train[config.col_train_seq] = list_train
del list_train
gc.collect()

X_train[config.col_target] = list_target
del list_target
gc.collect()

X_train.drop_duplicates(inplace=True)
# X_train.reset_index(drop=True, inplace=True)

X_train = pl.from_pandas(X_train)

  0%|          | 0/13265267 [00:00<?, ?it/s]

CPU times: user 44min 28s, sys: 1min 6s, total: 45min 35s
Wall time: 45min 28s


In [47]:
# 188131225

In [48]:
X_train.shape

(68570060, 3)

In [49]:
# X_train = X_train.filter(pl.col(config.col_target) != pad)
# X_train.shape

In [50]:
# %%time

# X_train = X_train.to_pandas()
# X_train['len'] = X_train[config.col_train_seq].progress_apply(len)
# X_train = X_train[X_train['len'] > 0]
# X_train.drop(['len'], axis=1, inplace=True)
# X_train = pl.from_pandas(X_train)

In [51]:
# X_train.shape

In [52]:
X_train.head()

user,train_seq,target
i64,list[i64],i64
10757568,[892818],5628
12788480,[476984],793352
13740480,"[927857, 643103, ... 57108]",57108
13740480,"[927857, 927857, ... 57108]",57108
13740480,"[927857, 927857, ... 57108]",890385


In [53]:
# X_train.shape

In [54]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].parallel_apply(lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False))
# X_train = pl.from_pandas(X_train)

In [55]:
# X_train.head()

In [56]:
# X_train.shape

In [57]:
# X_train[config.col_target].max(), pad  # (31207, 31211)

In [58]:
X_train[config.col_target].value_counts()

target,counts
i64,u32
415808,7
201088,36
193152,8
23744,23
835392,73
539776,4
187776,96
185216,45
334912,62


In [59]:
X_val_seq.head()

user,aid
i64,list[i64]
13368128,"[954978, 954978, 290966]"
12914944,"[760692, 243874, ... 144252]"
14102016,"[579011, 954978]"
13455680,"[496025, 844449, 844449]"
14117248,[628702]


In [60]:
X_val_seq = X_val_seq.to_pandas()
X_val_seq[config.col_train_seq] = X_val_seq[config.col_aid].parallel_apply(
    lambda x: utils.truncate_list(l=list(x), max_len=config.max_len, from_start=False)
)
X_val_seq = pl.from_pandas(X_val_seq)

In [61]:
X_val_seq.head()

user,aid,train_seq
i64,list[i64],list[i64]
13368128,"[954978, 954978, 290966]","[954978, 954978, 290966]"
12914944,"[760692, 243874, ... 144252]","[760692, 243874, ... 144252]"
14102016,"[579011, 954978]","[579011, 954978]"
13455680,"[496025, 844449, 844449]","[496025, 844449, 844449]"
14117248,[628702],[628702]


In [62]:
X_val = X_val.join(
    X_val_seq[[config.col_user, config.col_train_seq]], how="inner", on=config.col_user
)

In [63]:
X_val = X_val.rename({config.col_aid: config.col_target})

In [64]:
X_val.filter(pl.col(config.col_user) == 13833088)

user,target,train_seq
i64,list[i64],list[i64]
13833088,"[998481, 998481, 226025]","[261470, 300201, ... 513695]"


In [65]:
pad

954978

In [66]:
# df_train[col_aids].value_counts().sort("counts", reverse=True)

In [67]:
# df_train_clicks.drop([col_aids, "split"], axis=1, inplace=True)

In [68]:
# (12773, 4)

In [69]:
# X_train.head()

In [70]:
X_val.head()

user,target,train_seq
i64,list[i64],list[i64]
13534272,"[296142, 1043933, 1043933]","[152033, 537217, ... 537217]"
13937600,"[1333440, 1616754, 1333440]","[98081, 98081]"
13966144,"[240128, 594088, 1811390]","[798, 15777, ... 29356]"
13048320,"[1559894, 498884, 1559894]","[86075, 86075, ... 802855]"
14308928,"[1246648, 1234859, 1234859]","[11022, 480360, ... 641589]"


In [71]:
class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super(Attention, self).__init__()

        self.hidden_size = hidden_size
        self.batch_first = batch_first

        self.att_weights = nn.Parameter(
            torch.Tensor(1, hidden_size), requires_grad=True
        )

        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def get_mask(self):
        pass

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        # print(f"inputs: {inputs.size()}")

        # apply attention layer
        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0)  # (1, hidden_size) --> (hidden_size, 1)
            .unsqueeze(0)  # (1, hidden_size, 1)
            .repeat(batch_size, 1, 1),  # (batch_size, hidden_size, 1)
        )

        # print(f"weights: {weights.size()}")

        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)
        # print(f"attentions: {attentions.size()}")

        # create mask based on the sentence lengths
        mask = torch.ones(attentions.size(), requires_grad=True).to(device)
        # print(f"mask: {mask.size()}")
        if batch_size == 1:
            mask = mask.view(1, -1)
        for i, l in enumerate(lengths):  # skip the first sentence
            if l < max_len:
                mask[i, l:] = 0

        # apply mask and renormalize attention scores (weights)
        masked = attentions * mask
        _sums = masked.sum(-1).unsqueeze(-1)  # sums per row

        attentions = masked.div(_sums)

        # apply attention weights
        weighted = torch.mul(inputs, attentions.unsqueeze(-1).expand_as(inputs))
        # print(f"weighted: {weighted.size()}")

        # get the final fixed vector representations of the sentences
        representations = weighted.sum(1).squeeze()
        # print(f"representations: {representations.size()}")

        return representations, attentions

In [72]:
# min(600, round(1.6 * n_cat**0.56))

n_vocab = pad + 1
# hidden_dim = 128
# embedding_dim = min(600, round(1.6 * pad**0.56))
# embedding_dim = 100

In [73]:
class RRNN(nn.Module):
    def __init__(
        self,
        n_vocab=n_vocab,
        hidden_dim=config.hidden_dim,
        embedding_dim=config.embedding_dim,
        dropout=0.1,
    ):
        super(RRNN, self).__init__()
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(n_vocab, embedding_dim, sparse=False)
        self.hidden_dim = hidden_dim
        self.out_features_num = n_vocab

        self.lstm_desc = nn.GRU(
            embedding_dim,
            hidden_dim,
            dropout=dropout,
            num_layers=1,
            bidirectional=False,
            batch_first=True,
        )

        self.pooling_result_dimension = self.hidden_dim * 1
        self.dim_after_concat = self.pooling_result_dimension * 1  # + 6 + 14 + 19

        self.attention_descs = Attention(self.hidden_dim * 1, batch_first=True)
        self.bn1 = nn.BatchNorm1d(self.dim_after_concat)
        self.dropout1 = nn.Dropout(dropout, inplace=True)
        self.fc1 = nn.Linear(self.dim_after_concat, self.embedding_dim)
        self.dropout2 = nn.Dropout(dropout, inplace=True)
        self.fc2 = nn.Linear(self.embedding_dim, self.out_features_num)

        self.fc2.weight = self.embeddings.weight

    def _create_embedding_projection(
        self,
        cardinality,
        embed_size,
        add_missing=True,
        padding_idx=0,
        scale_grad_by_freq=False,
    ):
        add_missing = add_missing * 1
        return nn.Embedding(
            num_embeddings=cardinality + add_missing,
            embedding_dim=embed_size,
            padding_idx=padding_idx,
            scale_grad_by_freq=scale_grad_by_freq,
        )

    def forward(self, aids, lens, targets=None):
        batch_size = aids.shape[0]

        embedded_aids = self.embeddings(aids)
        pad_embed_pack = torch.nn.utils.rnn.pack_padded_sequence(
            embedded_aids, list(lens), batch_first=True, enforce_sorted=False
        )
        states_aids, hh_aids = self.lstm_desc(pad_embed_pack)

        states_aids, lens_unpacked = torch.nn.utils.rnn.pad_packed_sequence(
            states_aids, batch_first=True
        )

        lengths_aids = [x_.size(1) for x_ in states_aids]
        atten_aids, _ = self.attention_descs(states_aids, lengths_aids)
        atten_aids = atten_aids.view(batch_size, -1)

        features = torch.cat(
            [
                # subcategory_embed, category_embed, region_embed,
                atten_aids
            ],
            dim=-1,
        )
        features = self.bn1(features)
        features = self.dropout1(features)
        features = features.view(features.size(0), -1)
        features = self.fc1(features)
        features = self.dropout2(features)
        features = self.fc2(features)

        return features

In [74]:
%%time
X_train = X_train.to_pandas()

index = X_train.index
target = X_train[config.col_target]
user = X_train[config.col_user]

X_train = pd.DataFrame(X_train[config.col_train_seq].tolist(), index=index)
X_train.columns = ['pos' + '_' + str(pos) for pos in X_train.columns]
X_train[config.col_target] = target
X_train[config.col_user] = user
cols_pos = [col for col in X_train.columns if "pos" in col]

X_train.to_parquet('train_exp.parquet', index=False)

del X_train, index, target
gc.collect()

X_val = X_val.to_pandas()

X_val_to_dataset = pd.DataFrame(X_val[config.col_train_seq].tolist(), index=X_val.index)
X_val_to_dataset.columns = ['pos' + '_' + str(pos) for pos in X_val_to_dataset.columns]
X_val_to_dataset[config.col_target] = X_val[config.col_target]
X_val_to_dataset[config.col_user] = X_val[config.col_user]

X_val_target_to_dataset = pd.DataFrame(X_val_to_dataset[config.col_target].tolist(), index=X_val_to_dataset.index)
X_val_target_to_dataset.columns = ['target' + '_' + str(pos) for pos in X_val_target_to_dataset.columns]
X_val_to_dataset[X_val_target_to_dataset.columns] = X_val_target_to_dataset[X_val_target_to_dataset.columns]
X_val_to_dataset.drop([config.col_target], axis=1, inplace=True)

# X_train = pl.from_pandas(X_train)
X_val = pl.from_pandas(X_val)

cols_target_val = [col for col in X_val_to_dataset.columns if "target" in col]

X_val_to_dataset.to_parquet('valid_exp.parquet', index=False)
del X_val_to_dataset
gc.collect()

CPU times: user 3min 26s, sys: 26.3 s, total: 3min 53s
Wall time: 3min 49s


0

In [75]:
cols_target_val = ["target_0", "target_1", "target_2"]
cols_pos = [
    "pos_0",
    "pos_1",
    "pos_2",
    "pos_3",
    "pos_4",
    "pos_5",
    "pos_6",
    "pos_7",
    "pos_8",
    "pos_9",
]

In [76]:
# %%time

# X_train_to_dataset.to_parquet('train.parquet', index=False)
# X_val_to_dataset.to_parquet('valid.parquet', index=False)

In [77]:
%%time

train_ds = Dataset_merlin('train_exp.parquet')
train_dl_merlin = Loader(dataset=train_ds, batch_size=config.batch_size_train, shuffle=True)

valid_ds = Dataset_merlin('valid_exp.parquet')
valid_dl_merlin = Loader(dataset=valid_ds, batch_size=config.batch_size_infer, shuffle=False)

CPU times: user 12.6 s, sys: 12.6 s, total: 25.2 s
Wall time: 3.92 s


In [78]:
import math


def get_matrix_and_target_from_batch(
    batch, cols_pos=cols_pos, cols_target_val=cols_target_val, pad=pad, mod="inference"
):
    assert mod in ["inference", "train", "valid"]

    batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
    for i in range(len(cols_pos)):
        batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
    batch_matrix = [
        torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
        for i in range(len(batch_matrix))
    ]
    lens = torch.tensor(list(map(len, batch_matrix)))

    batch_matrix = torch.nn.utils.rnn.pad_sequence(
        batch_matrix, batch_first=True, padding_value=pad
    )

    users = batch[config.col_user].squeeze(1).tolist()

    if mod == "train":
        target = batch[config.col_target]
    elif mod == "valid":
        target = torch.zeros([len(batch[cols_target_val[0]]), len(cols_target_val)])
        for i in range(len(cols_target_val)):
            target[:, i] = torch.reshape(batch[cols_target_val[i]], (1, -1))
        target = [
            torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
            for i in range(len(target))
        ]
    elif mod == "inference":
        target = None
    return batch_matrix, target, lens, users

In [79]:
# def get_matrix_and_target_from_batch(
#     batch, cols_pos=cols_pos, cols_target_val=cols_target_val, pad=pad, train=True
# ):
#     batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
#     for i in range(len(cols_pos)):
#         batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
#     batch_matrix = [
#         torch.tensor([int(val) for val in batch_matrix[i] if not math.isnan(val)])
#         for i in range(len(batch_matrix))
#     ]
#     lens = torch.tensor(list(map(len, batch_matrix)))

#     batch_matrix = torch.nn.utils.rnn.pad_sequence(
#         batch_matrix, batch_first=True, padding_value=pad
#     )

#     if train:
#         target = batch[config.col_target]
#     else:
#         target = torch.zeros([len(batch[cols_target_val[0]]), len(cols_target_val)])
#         for i in range(len(cols_target_val)):
#             target[:, i] = torch.reshape(batch[cols_target_val[i]], (1, -1))
#         target = [
#             torch.tensor([int(val) for val in target[i] if not math.isnan(val)])
#             for i in range(len(target))
#         ]
#     return batch_matrix, target, lens

In [80]:
# del df, X_train, X_train_to_dataset, X_val_to_dataset
# gc.collect()

In [81]:
# %%time

# for batch, _ in tqdm(train_dl_merlin):
#     # batch_matrix, batch["val"], lens = get_matrix_and_target_from_batch(batch, cols_pos=cols_pos, pad=pad)
#     continue
#     # break
# # 5 min

In [82]:
# %%time

# batch_matrix = np.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
# for i in range(len(cols_pos)):
#     batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
# batch_matrix = [
#         torch.tensor([int(val) for val in batch_matrix[i] if not np.isnan(val)])
#         for i in range(len(batch_matrix))
#     ]

In [83]:
# def get_matrix_and_target_from_batch(batch, cols_pos=cols_pos):
#     batch_matrix = torch.zeros([len(batch[cols_pos[0]]), len(cols_pos)])
#     for i in range(len(cols_pos)):
#         batch_matrix[:, i] = torch.reshape(batch[cols_pos[i]], (1, -1))
#     # batch_target = torch.reshape(batch["val"], (1, -1))
#     return batch_matrix, batch["val"]

In [85]:
# del X_train
# gc.collect()

In [86]:
DEBUG = False


def do_inverse_transform(array, ordinal_encoder=ordinal_encoder):
    result = ordinal_encoder.inverse_transform(array.reshape(-1, 1)).reshape(1, -1)[0]
    result = [val for val in result if val is not None]
    return result


def flatten_list(l):
    return [item for sublist in l for item in sublist]


def valid_func(valid_loader, model, criterion):
    model.eval()

    bar = tqdm(valid_loader)
    PREDS = list()
    TARGETS = list()
    list_users = list()
    losses = list()
    with torch.no_grad():
        for batch_idx, (batch, _) in enumerate(bar):
            aids, targets, lens, users = get_matrix_and_target_from_batch(
                batch, mod="valid"
            )
            aids = aids.to(device).long()
            lens = lens.to(device).long()
            # category = category.to(device).long()
            # subcategory = subcategory.to(device).long()
            # region = region.to(device).long()
            # targets = targets.to(device).long()
            # targets = targets.squeeze(1)

            logits = model(aids, lens)
            # loss = criterion(logits, targets)
            # losses.append(loss.item())

            # PREDS += [torch.argmax(logits, 1).detach().cpu()]
            k = 20
            preds_cur = torch.topk(logits, k).indices.detach().cpu().numpy()
            # print(preds_cur.shape)
            # print(preds_cur)
            # preds_cur = [np.argpartition(-pred, k)[:k] for pred in preds_cur]
            # preds_cur = [
            #     do_inverse_transform(pred, ordinal_encoder=ordinal_encoder)
            #     for pred in preds_cur
            # ]
            PREDS.append(preds_cur)
            targets = [target.tolist() for target in targets]
            TARGETS.append(targets)
            list_users.append(users)
            # bar.set_description(f"loss: {loss.item():.5f}")

    # loss_valid = np.mean(losses)
    # print("loss_valid:", loss_valid)
    list_users = utils.flatten_list(list_users)
    # PREDS = torch.cat(PREDS).cpu().tolist()
    PREDS = utils.flatten_list(PREDS)
    TARGETS = utils.flatten_list(TARGETS)
    # TARGETS = torch.cat(TARGETS).cpu().tolist()
    # print(len(PREDS), len(TARGETS))
    # print(TARGETS)
    # print(PREDS)
    df_val = pd.DataFrame(
        {"idx": np.arange(len(list_users)), "session": list_users, "preds": PREDS}
    )
    df_val = df_val.explode("preds")

    df_val["preds"] = ordinal_encoder.inverse_transform(
        df_val["preds"].values.reshape(-1, 1)
    ).reshape(1, -1)[0]
    df_val = df_val.dropna()
    df_val["preds"] = df_val["preds"].astype(int)
    df_val = pl.from_pandas(df_val)
    df_val = (
        (
            df_val.groupby(["session", "idx"]).agg(
                [
                    pl.col("preds").list(),
                ]
            )
        )
        .sort("idx")
        .to_pandas()
    )

    PREDS = df_val["preds"].tolist()
    PREDS = [list(pred) for pred in PREDS]
    df_val["targets"] = TARGETS

    metric_total = utils.recall_several_targets(PREDS, TARGETS)
    print("metric_total:", metric_total)

    return df_val


# def inference_func(inference_loader, model):
#     model.eval()

#     bar = tqdm(inference_loader)
#     preds = list()

#     with torch.no_grad():
#         for batch_idx, (descs, category, subcategory, region) in enumerate(bar):
#             descs = descs.to(device).long()
#             category = category.to(device).long()
#             subcategory = subcategory.to(device).long()
#             region = region.to(device).long()

#             logits = model(descs, category, subcategory, region)

#             preds += [logits.detach().cpu()]

#     preds = torch.cat(preds).cpu().numpy()
#     preds = 1.0 / (1.0 + np.exp(-preds))
#     preds = np.concatenate(preds, axis=0)
#     return preds


def train_func(
    train_loader,
    model,
    criterion,
    opts: list,
    amp=False,
    num_batches_to_check=None,
    num_batches_to_validate=None,
    valid_loader=None,
):
    model.train()
    bar = tqdm(train_loader)
    losses = list()
    if amp:
        scaler = torch.cuda.amp.GradScaler()

    start_time = time.perf_counter()
    batches_passed = 0
    for batch_idx, (batch, _) in enumerate(bar):
        if num_batches_to_check:
            if batch_idx % num_batches_to_check == 0 and batch_idx > 0:
                batches_passed += num_batches_to_check
                end_time = time.perf_counter()
                total_time = end_time - start_time
                print(f"batches passed: {batches_passed}")
                print(
                    f"last {num_batches_to_check} batches took {total_time / 60 :.4f} minutes"
                )
                print()
                start_time = time.perf_counter()

        aids, targets, lens, users = get_matrix_and_target_from_batch(
            batch, mod="train"
        )
        aids = aids.to(device).long()
        lens = lens.to(device).long()
        # category = category.to(device).long()
        # subcategory = subcategory.to(device).long()
        # region = region.to(device).long()

        targets = targets.to(device).long()
        targets = targets.squeeze(1)

        if DEBUG and batch_idx == 2:
            print("Debug Mode. Only train on first 1 batches.")
            break

        if amp:
            with torch.cuda.amp.autocast():
                logits = model(aids, lens)
                loss = criterion(logits)
                scaler.scale(loss).backward()  # loss.backward()
                for optimizer in opts:
                    scaler.step(optimizer)  # optimizer.step()
                    scaler.update()
        else:
            logits = model(aids, lens)
            # print(logits.size())
            loss = criterion(logits, targets)
            loss.backward()
        for optimizer in opts:
            optimizer.step()

        for optimizer in opts:
            optimizer.zero_grad()

        losses.append(loss.item())

        if num_batches_to_validate:
            if batch_idx % num_batches_to_validate == 0 and batch_idx > 0:
                print("batch_idx:", batch_idx)
                model.eval()
                preds_local = valid_func(valid_loader, model, criterion)
                model.train()
                print("_" * 10)
                print()
        # smooth_loss = np.mean(losses[-30:])

        # bar.set_description(f"smooth_loss: {smooth_loss:.5f}")

    loss_train = np.mean(losses)
    print("loss_train:", loss_train)
    return loss_train

In [87]:
out_features_num = pad  # + 1
out_features_num

954978

In [88]:
model = RRNN()

model.load_state_dict(
    torch.load(
        f"{config.model_name}_epoch{9}_larger_lr_and_bs_and_train.pth",
    )
)

model.to(device)

/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.


/usr/bin/nvidia-modprobe: unrecognized option: "-f"

ERROR: Invalid commandline, please run `/usr/bin/nvidia-modprobe --help` for usage information.




RRNN(
  (embeddings): Embedding(954979, 100)
  (lstm_desc): GRU(100, 128, batch_first=True, dropout=0.1)
  (attention_descs): Attention()
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.1, inplace=True)
  (fc1): Linear(in_features=128, out_features=100, bias=True)
  (dropout2): Dropout(p=0.1, inplace=True)
  (fc2): Linear(in_features=100, out_features=954979, bias=True)
)

In [89]:
params_sparsed = list()
params_not_sparsed = list()
cnt = 0
for param in model.parameters():
    if cnt == 0:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)
    cnt += 1

In [90]:
params_sparsed[0].shape

torch.Size([954979, 100])

In [91]:
class MultipleOptimizer:
    def __init__(self, *optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for optimizer in self.optimizers:
            optimizer.zero_grad()

    def step(self):
        for optimizer in self.optimizers:
            optimizer.step()


class MultipleScheduler:
    def __init__(self, *schedulers):
        self.schedulers = schedulers

    def step(self):
        for scheduler in self.schedulers:
            scheduler.step()

In [92]:
init_lr = config.init_lr

criterion = nn.CrossEntropyLoss(ignore_index=pad)
optimizer = optim.Adam(model.parameters(), lr=init_lr)
optimizer_sparsed = optim.SparseAdam(params_sparsed, lr=init_lr)
optimizer_not_sparsed = optim.Adam(params_not_sparsed, lr=init_lr)
opts = [optimizer]  # , optimizer_sparsed, optimizer_not_sparsed
opt = MultipleOptimizer(optimizer_sparsed, optimizer_not_sparsed)

# scheduler_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler_not_sparsed = torch.optim.lr_scheduler.CosineAnnealingLR(
#     optimizer_sparsed, n_epochs
# )
# scheduler = MultipleScheduler(scheduler_sparsed, scheduler_not_sparsed)

seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [93]:
%%time

for epoch in tqdm(range(config.n_epochs), desc="epochs"):

    print(f"Epoch {epoch + 1} started")
    torch.cuda.empty_cache()

    # print("LR:", scheduler.schedulers[0].state_dict()["_last_lr"][0])
    loss_train = train_func(train_dl_merlin, model, criterion, opts, num_batches_to_validate=10_000, valid_loader=valid_dl_merlin)
    # scheduler.step()

    df_val = valid_func(valid_dl_merlin, model, criterion)
    # X_val['preds'] = preds_local
    # aucs = list()
    # for category, cur_df in X_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('local mean_auc:', np.mean(aucs))

    # loss_valid, preds = valid_func(valid_loader, model, criterion)
    # df_val['preds'] = preds
    # aucs = list()
    # for category, cur_df in df_val.groupby('category'):
    #     cur_auc = roc_auc_score(cur_df['is_bad'], cur_df['preds'])
    #     aucs.append(cur_auc)
    # print()
    # print(aucs)
    # print('mean_auc:', np.mean(aucs))

    torch.save(
        model.state_dict(),
        f"{config.model_name}_epoch{9 + epoch + 1}_larger_lr_and_bs_exp.pth",
    )

    print("Checkpoint saved!")
    print()
    print("*" * 90)
    print()
    print()

# del model
torch.cuda.empty_cache()
gc.collect()

epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 started


  0%|          | 0/133926 [00:00<?, ?it/s]

batch_idx: 10000


  0%|          | 0/912 [00:00<?, ?it/s]

metric_total: 0.3026423177850372
__________

batch_idx: 20000


... (24 outputs left out)

  0%|          | 0/912 [00:00<?, ?it/s]

metric_total: 0.3054949213659651
Checkpoint saved!

******************************************************************************************


Epoch 2 started


  0%|          | 0/133926 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df_val = valid_func(valid_dl_merlin, model, criterion)

In [ ]:
df_val

In [ ]:
val["len_seq"] = val[config.col_train_seq].apply(len)

In [ ]:
# metric_total: 0.042083333333333334 1 ep
# metric_total: 0.06625 1 ep 5 iter
# metric_total: 0.059583333333333335 1 ep 10 iter
# metric_total: 0.0725 1 ep 15 iter

In [ ]:
X_val[config.col_target].value_counts().head(10)

In [ ]:
X_train["val"].value_counts().head(10)

In [ ]:
X_val = X_val.to_pandas()

In [ ]:
X_val["preds"] = preds_local
X_val["recall_row"] = X_val.progress_apply(
    lambda x: utils.recall_several_targets_one_row(
        x["preds"], x[config.col_target].tolist()
    ),
    axis=1,
)

In [ ]:
X_val.shape

In [ ]:
X_val.tail()

In [ ]:
stop

In [ ]:
X_val[X_val["recall_row"] == 1].shape

In [ ]:
X_val_true = X_val[X_val["recall_row"] == 1]

In [ ]:
pad

In [ ]:
X_val_true.head()

In [ ]:
X_train[X_train["val"] == pad].shape[0] / X_train.shape[0]

In [ ]:
X_val_true.iloc[0]["preds"]

In [ ]:
stop
# loss_train: 12.190348625183105

# loss_train: 9.231197555859884
# loss_valid: 20.559587955474854
# metric_total: 0.000999000999000999

In [ ]:
params_sparsed = list()
params_not_sparsed = list()
for param in model.parameters():
    if param.grad.is_sparse:
        params_sparsed.append(param)
    else:
        params_not_sparsed.append(param)

In [ ]:
params_sparsed

In [ ]:
df_train.groupby(["n_week"]).count().sort("n_week", reverse=False)

In [ ]:
df_train_ex = df_train.filter(pl.col("user") <= 10000)

In [ ]:
%%time

dict_train = dict()
dict_answers = dict()
dict_val_len = dict()
dict_user = dict()
key = 1

for df_user in tqdm(df_train.groupby(["user"], maintain_order=True)):
    df_weeks_count = df_user.groupby(["n_week"]).count().sort("n_week", reverse=False)
    if len(df_weeks_count) <= 1:
        continue
    user = df_user["user"][0]
    for i in range(len(df_weeks_count["n_week"]) - 1):
        n_week_cur = df_weeks_count["n_week"][i]
        val_len = 34 - n_week_cur
        dict_train[key] = df_user.filter(pl.col("n_week") == n_week_cur)
        dict_answers[key] = df_user.filter(pl.col("n_week") > n_week_next)
        dict_val_len[key] = val_len
        dict_user[key] = user
        key += 1

In [ ]:
dict_train[1]

In [ ]:
dict_answers[1]

In [ ]:
df_user.filter(pl.col("n_week") == n_week_cur)

In [ ]:
for i in range(len(df_weeks_count["n_week"]) - 1):
    n_week_cur = df_weeks_count["n_week"][i]
    n_week_next = df_weeks_count["n_week"][i + 1]
    print(n_week_cur, n_week_next)

In [ ]:
df_user

In [ ]:
df_train_ex

In [ ]:
df_train.tail(10)

In [ ]:
df_test.head()

In [ ]:
utils.plot_user_activity(user=12899774, df=df_train)

In [ ]:
df_train["type"].value_counts()

In [ ]:
df_train["date"].min(), df_train["date"].max()

In [ ]:
df_test["date"].min(), df_test["date"].max()

In [ ]:
%%time

df_train.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
%%time

df_test.groupby(['dayofweek', 'type'])[['ts']].count()

In [ ]:
df_train_n_session = (df_train.groupby("user")[["n_session"]].max() + 1).reset_index()
df_test_n_session = (df_test.groupby("user")[["n_session"]].max() + 1).reset_index()

In [ ]:
df_train_n_session["n_session"].value_counts()

In [ ]:
df_test_n_session["n_session"].value_counts()

In [ ]:
def plot_n_interactions_per_dayofweek(df):
    try:
        df_name = df.name
    except AttributeError:
        df_name = None

    title = "n_interactions per dayofweek"
    if df_name is not None:
        title += f" in {df_name}"

    plt.figure(figsize=(12, 7))
    plt.title(title)
    plt.xlabel("dayofweek")
    plt.ylabel("n_interactions")

    dict_type_inter_offset = dict()
    dict_type_inter_offset["clicks"] = -0.2
    dict_type_inter_offset["carts"] = 0.0
    dict_type_inter_offset["orders"] = 0.2

    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "g"
    dict_type_inter_color["orders"] = "r"

    for (dayofweek, type_inter), df_cur in df.groupby(["dayofweek", "type"]):
        ys = df_cur["ts"].count()
        plt.bar(
            dayofweek + dict_type_inter_offset[type_inter],
            ys,
            width=0.2,
            color=dict_type_inter_color[type_inter],
            align="center",
            # label=feature,
            # marker=",",
            # linestyle="dashed",
            alpha=0.6,
        )

In [ ]:
plot_n_interactions_per_dayofweek(df_train)

In [ ]:
plot_n_interactions_per_dayofweek(df_test)

In [ ]:
def plot_user_activity(user, df):
    dict_type_inter_color = dict()
    dict_type_inter_color["clicks"] = "b"
    dict_type_inter_color["carts"] = "orange"
    dict_type_inter_color["orders"] = "r"

    offset = 0.5
    df_cur = df[df["user"] == user]
    df_cur["date_day"] = df_cur["date_day"].astype(str)

    if df.name == "df_train":
        x = pd.date_range("2022-08-01", "2022-08-28")
    elif df.name == "df_test":
        x = pd.date_range("2022-08-29", "2022-09-04")
    x = [str(d).split(" ")[0] for d in x]
    map_dates = {x[i]: i for i in range(len(x))}
    points_sundays = [point for point in range(len(x)) if (point + 1) % 7 == 0]

    plt.figure(figsize=(12, 7))
    plt.xticks(range(len(x)), x, rotation=90)
    plt.yticks(range(25))

    for point in range(25):
        plt.axhline(y=point, color="gray", linestyle="-", alpha=0.1)
    for point in range(len(x)):
        plt.axvline(x=point + offset, color="gray", linestyle="-", alpha=0.1)
        plt.axvline(x=point - offset, color="gray", linestyle="-", alpha=0.1)
    for point in points_sundays:
        plt.axvline(x=point + offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset, color="r", linestyle="-", alpha=0.3)
        plt.axvline(x=point - offset - 1, color="r", linestyle="-", alpha=0.3)

    for action, df_cur_action in df_cur.groupby("type"):
        xx = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        yy = np.random.uniform(-0.15, 0.15, len(df_cur_action))
        plt.scatter(
            df_cur_action["date_day"].map(map_dates) + xx,
            df_cur_action["hour"] + yy,
            c=dict_type_inter_color[action],
            s=10,
            label=action,
        )
    plt.legend(
        loc="upper right", bbox_to_anchor=(1.1, 1.0), ncol=1, fancybox=True, shadow=True
    )

In [ ]:
plot_user_activity(user=2932269, df=df_train)

In [ ]:
plot_user_activity(user=12899780, df=df_test)